In [2]:
# =====================================================
# ✅ 이진분류 4개 모델 비교 (NaN 안전 처리 포함 완성본)
# - Logistic / LinearSVC+Calibrated / XGBoost / LightGBM
# - 시간 기준 split / 원핫 통일 / 날씨 변수 포함
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)

# xgboost (선택)
try:
    from xgboost import XGBClassifier
    _HAS_XGB = True
except Exception as e:
    _HAS_XGB = False
    print("⚠️ xgboost import 실패:", e)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


# =========================
# 설정
# =========================
csv_path = "new_flight_analysis_summary.csv"
THR = 0.6
THS = np.unique(np.r_[np.linspace(0.05, 0.95, 19), THR])
THS.sort()


# =====================================================
# 1) 데이터 로드 + datetime + 타깃
# =====================================================
df = pd.read_csv(csv_path)
print("✅ CSV 로드:", len(df))

# datetime 컬럼 자동 선택
if "departure_datetime" in df.columns:
    dt_col = "departure_datetime"
elif "일시" in df.columns:
    dt_col = "일시"
else:
    raise KeyError("❌ datetime 컬럼 없음")

df[dt_col] = pd.to_datetime(df[dt_col], errors="coerce")
df = df[df[dt_col].notna()].copy()

# 시간 파생
df["dep_hour"] = df[dt_col].dt.hour
df["dep_minute"] = df[dt_col].dt.minute
df["dep_weekday"] = df[dt_col].dt.weekday
df["is_weekend"] = df["dep_weekday"].isin([5, 6]).astype(int)

# arrival_code
if "arrival_code" in df.columns:
    df["arrival_code"] = pd.to_numeric(df["arrival_code"], errors="coerce").fillna(-1).astype(int)

# 타깃
df["is_delay"] = df["is_delay"].astype(int)

print("✅ 양성 비율:", df["is_delay"].mean().round(4))


# =====================================================
# 2) 시간 기준 Train/Test split
# =====================================================
df = df.sort_values(dt_col)
split_dt = df[dt_col].quantile(0.8)

train_df = df[df[dt_col] <= split_dt]
test_df  = df[df[dt_col] > split_dt]

X_train = train_df
y_train = train_df["is_delay"].values

X_test = test_df
y_test = test_df["is_delay"].values


# =====================================================
# 3) 피처 정의 (날씨 포함)
# =====================================================
num_cols = [
    "dep_hour",
    "dep_minute",
    "dep_weekday",
    "is_weekend",
    "기온(°C)",
    "풍속_ms"
]
num_cols = [c for c in num_cols if c in df.columns]

cat_cols = ["항공사", "출발지", "arrival_code", "flight_type"]
cat_cols = [c for c in cat_cols if c in df.columns]

print("✅ num_cols:", num_cols)
print("✅ cat_cols:", cat_cols)


# =====================================================
# 4) 전처리 (NaN 안전)
# =====================================================
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
            ("ohe", ohe)
        ]), cat_cols),

        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols)
    ],
    remainder="drop"
)

# 불균형 가중치
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
scale_pos_weight = neg / max(pos, 1)


# =====================================================
# 5) 모델 정의
# =====================================================
models = {}

models["Logistic"] = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

base_linear = Pipeline([
    ("prep", preprocessor),
    ("clf", LinearSVC(class_weight="balanced", random_state=42))
])
models["LinearSVC(Calibrated)"] = CalibratedClassifierCV(
    base_linear, cv=3, method="sigmoid"
)

if _HAS_XGB:
    models["XGBoost"] = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=700,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        ))
    ])

models["LightGBM"] = Pipeline([
    ("prep", preprocessor),
    ("clf", LGBMClassifier(
        n_estimators=900,
        learning_rate=0.05,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

print("✅ 모델:", list(models.keys()))


# =====================================================
# 6) 학습 + 확률 예측
# =====================================================
probs = {}
for name, model in models.items():
    print("\nTRAIN:", name)
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X_test)[:, 1]
    else:
        s = model.decision_function(X_test)
        p = (s - s.min()) / (s.max() - s.min() + 1e-9)

    probs[name] = p


# =====================================================
# 7) 성능 비교 표
# =====================================================
rows = []
for name, p in probs.items():
    pred = (p >= THR).astype(int)
    rows.append([
        name, THR,
        roc_auc_score(y_test, p),
        average_precision_score(y_test, p),
        accuracy_score(y_test, pred),
        precision_score(y_test, pred, zero_division=0),
        recall_score(y_test, pred, zero_division=0),
        f1_score(y_test, pred, zero_division=0)
    ])

leaderboard = pd.DataFrame(
    rows, columns=["model","thr","ROC_AUC","PR_AUC","ACC","PREC","RECALL","F1"]
).sort_values(["PR_AUC","F1","ROC_AUC"], ascending=False)

display(leaderboard)


✅ CSV 로드: 2836872
✅ 양성 비율: 0.1701
✅ num_cols: ['dep_hour', 'dep_minute', 'dep_weekday', 'is_weekend']
✅ cat_cols: ['항공사', '출발지', 'arrival_code', 'flight_type']
✅ 모델: ['Logistic', 'LinearSVC(Calibrated)', 'XGBoost', 'LightGBM']

TRAIN: Logistic

TRAIN: LinearSVC(Calibrated)

TRAIN: XGBoost

TRAIN: LightGBM
[LightGBM] [Info] Number of positive: 336168, number of negative: 1933330
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011573 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 387
[LightGBM] [Info] Number of data points in the train set: 2269498, number of used features: 151
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.148124 -> initscore=-1.749388
[LightGBM] [Info] Start training from score -1.749388


C:\Users\Admin\anaconda3\envs\4vector\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,model,thr,ROC_AUC,PR_AUC,ACC,PREC,RECALL,F1
3,LightGBM,0.6,0.739064,0.488972,0.733358,0.481577,0.450511,0.465526
2,XGBoost,0.6,0.711470,0.451563,0.719307,0.448047,0.383692,0.413380
1,LinearSVC(Calibrated),0.6,0.661088,0.373362,0.742242,0.000000,0.000000,0.000000
0,Logistic,0.6,0.658297,0.370077,0.699734,0.393031,0.302971,0.342174
